In [8]:
import torch
from transformers import DistilBertTokenizer, AutoModelForMaskedLM

In [131]:
def get_filtered_logits(text, model_name="distilbert-base-uncased", model_instance=None):
    """
    Processes input text through DistilBERT and removes padding and special tokens from logits.
    """
    # Use provided model instance or create a new one
    if model_instance is None:
        tokenizer = DistilBertTokenizer.from_pretrained(model_name)
        model = AutoModelForMaskedLM.from_pretrained(model_name, num_labels=2)
    else:
        model = model_instance
        tokenizer = DistilBertTokenizer.from_pretrained(model_name)

    # Tokenize input text
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    # Extract attention mask and special tokens mask
    attention_mask = inputs["attention_mask"]  # 1 for real tokens, 0 for padding
    special_tokens_mask = tokenizer.get_special_tokens_mask(
        inputs["input_ids"][0].tolist(), already_has_special_tokens=True
    )
    special_tokens_mask = torch.tensor(special_tokens_mask).unsqueeze(0)  # Convert to tensor and add batch dim

    # Get logits from the model
    output = model(**inputs)

    logits = output.logits  # Shape: (batch_size, seq_len, num_labels)

    # Remove padding tokens and special tokens
    logits = logits * attention_mask.unsqueeze(-1) * (1 - special_tokens_mask).unsqueeze(-1)

    return logits

def spladeify_logits(logits):
    transformed_logits = torch.log1p(torch.relu(logits))
    pooled = torch.max(transformed_logits, dim=1).values
    return pooled

def splade(text):
    logits = get_filtered_logits(text)
    splade_output = spladeify_logits(logits)
    return splade_output

def get_top_tokens(vector, tokenizer_name="distilbert-base-uncased", top_n=10):
    tokenizer = DistilBertTokenizer.from_pretrained(tokenizer_name)
    if not isinstance(vector, torch.Tensor):
        vector = torch.tensor(vector)
        
    top_indices = torch.topk(vector, top_n).indices.tolist()
    top_tokens = [tokenizer.convert_ids_to_tokens(idx) for idx in top_indices]
    return top_tokens


test_splade_rep = splade("My name is Jack!")

In [132]:
def loss_func(splade_input):
    target_token = 2990
    loss_value = torch.abs(splade_input[:, target_token]).sum()
    return -loss_value
    
loss_func(test_splade_rep)

tensor(-3.0410, grad_fn=<NegBackward0>)

In [137]:
def train(text, steps=100):
    # Load tokenizer and model once
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForMaskedLM.from_pretrained("distilbert-base-uncased", num_labels=2)
    
    # Initialize optimizer with model parameters
    optim = torch.optim.Adam(model.parameters(), lr=0.01)
    
    # Create a version of splade that uses our specific model instance
    def splade_with_model(input_text):
        return spladeify_logits(get_filtered_logits(input_text, model_instance=model))
    
    for step in range(steps):
        # Zero gradients
        optim.zero_grad()
        
        # Forward pass with our specific model
        model_out = splade_with_model(text)
        
        # Calculate loss (minimize activation for token 2990)
        loss = loss_func(model_out)
        
        # Backpropagation
        loss.backward()
        optim.step()
    print(f"Step {step}, Loss value: {loss.item()}")
    print(get_top_tokens(model_out.detach().numpy()))

In [138]:
output = train("My name is Jack")

Step 99, Loss value: -0.0
[['.', 'is', ';', 'as', 'nearest', ',', 'from', 'nor', 'natural', '##sms']]


In [119]:
DistilBertTokenizer.from_pretrained("distilbert-base-uncased")("My name is Jack")

{'input_ids': [101, 2026, 2171, 2003, 2990, 102], 'attention_mask': [1, 1, 1, 1, 1, 1]}

In [128]:
output[:,2990]

array([-0.], dtype=float32)